<a href="https://colab.research.google.com/github/t-roscommon/hea-material-discovery/blob/main/hea_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HEA Catalyst Screening Pipeline
### Initialization
Imports necessary packages, initializes sorted element pool, and then defines some helper functions used later, largely for data generation and cleaning.

In [ ]:
!pip install AlloySustainability pymatgen matminer
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from pymatgen.core import Composition, Element
from matminer.featurizers.composition import ElementProperty, ValenceOrbital, AtomicOrbitals
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

from AlloySustainability.computations import (
    load_element_indicators,
    load_RTHEAs_vs_Fe_df,
    load_HTHEAs_vs_Ni_df,
    compute_impacts
)
from AlloySustainability.visualization import plot_alloy_comparison
import os
import zipfile
import seaborn as sns
from IPython.display import FileLink, display

# Fixed element order required by the package
element_pool = [
    'Al', 'Co', 'Cr', 'Cu', 'Fe', 'Hf',
    'Mn', 'Mo', 'Nb', 'Ni', 'Re', 'Ru',
    'Si', 'Ta', 'Ti', 'V',  'W',  'Zr'
]

# --- Featurizers ---
featurizer    = ElementProperty.from_preset("magpie")
vo_featurizer = ValenceOrbital(props=['avg', 'frac'])
ao_featurizer = AtomicOrbitals()
for f in [featurizer, vo_featurizer, ao_featurizer]:
    f.set_n_jobs(1)

# --- Helper functions ---
def safe_composition(formula):
    try:
        return Composition(formula)
    except Exception:
        return None

def safe_featurize(f, c):
    try:
        return f.featurize(c)
    except Exception:
        return [None] * len(f.feature_labels())

# edited so each percentage is in multiples of 5
def generate_random_composition(element_pool, min_elements=4, max_elements=6):
    n = np.random.randint(min_elements, max_elements + 1)
    elements = np.random.choice(element_pool, n, replace=False)

    while True:
        cuts = sorted(np.random.choice(range(5, 100, 5), n - 1, replace=False))
        cuts = [0] + cuts + [100]
        percentages = [cuts[i+1] - cuts[i] for i in range(n)]
        if all(p >= 5 for p in percentages):
            break

    fractions = [p / 100 for p in percentages]
    return dict(zip(elements, fractions))

def composition_to_formula(comp_dict):
    counts = {el: max(1, round(f * 100)) for el, f in comp_dict.items()}
    return ''.join([f"{el}{count}" for el, count in counts.items()])

def to_mass_fractions(comp_dict):
    """Convert mole fractions -> mass fractions ordered by ELEMENT_ORDER."""
    masses = {
        el: float(Element(el).atomic_mass) * frac
        for el, frac in comp_dict.items()
    }
    total = sum(masses.values())
    return [masses.get(el, 0.0) / total for el in element_pool]

### Composition Generation and MatMiner Featurization
From a pool of 18 transition metals (the ones included in the AlloySustainability dataset), generates 100k compounds with 4, 5, or 6 elements and random at%'s that add up to 100%. Then, concatenates the table into a one-column-wide vector and featurizes each compound.

In [ ]:
n_samples    = 100_000
compositions = [generate_random_composition(element_pool) for _ in range(n_samples)]

df = pd.DataFrame({
    'elements': compositions,
    'formula':  [composition_to_formula(c) for c in compositions]
})

df['composition'] = df['formula'].apply(safe_composition)
df = df[df['composition'].notna()].reset_index(drop=True)

# --- Featurize --- (takes a few minutes)
magpie_cand = pd.DataFrame(
    df['composition'].apply(featurizer.featurize).tolist(),
    columns=featurizer.feature_labels()
)
vo_cand = pd.DataFrame(
    df['composition'].apply(lambda c: safe_featurize(vo_featurizer, c)).tolist(),
    columns=vo_featurizer.feature_labels()
)
ao_cand = pd.DataFrame(
    df['composition'].apply(lambda c: safe_featurize(ao_featurizer, c)).tolist(),
    columns=ao_featurizer.feature_labels()
)

# Drop non-numeric AtomicOrbitals columns
ao_cand = ao_cand.apply(pd.to_numeric, errors='coerce')
ao_cand = ao_cand.loc[:, ao_cand.notna().all()]

# --- Assemble ---
df_featurized = pd.concat([df, magpie_cand, vo_cand, ao_cand], axis=1)

print(f"Featurized shape: {df_featurized.shape}")
print(f"NaN rows: {df_featurized.isna().any(axis=1).sum()}")

### Fetch, clean, featurize training data

Fetch, clean, and featurize training data. Fetch N* adsorption training labels via Catalysis-Hub GraphQL API, then featurizes the data based on several electrochemical characteristics and plots a histogram displaying the frequency of compounds per ΔGN value.

In [ ]:
# queries API for bimetallic adsorption training data
all_edges    = []
after_cursor = None

while True:
    after_clause = f', after: "{after_cursor}"' if after_cursor else ''
    query = f"""
    {{
      reactions(pubId: "MamunHighT2019", first: 1000{after_clause}) {{
        pageInfo {{ hasNextPage endCursor }}
        edges {{
          node {{
            surfaceComposition
            facet
            reactionEnergy
            reactants
            products
          }}
        }}
      }}
    }}
    """
    data = requests.post(
        'https://api.catalysis-hub.org/graphql',
        json={'query': query}
    ).json()['data']['reactions']

    all_edges.extend(data['edges'])
    if not data['pageInfo']['hasNextPage']:
        break
    after_cursor = data['pageInfo']['endCursor']

print(f"Total reactions fetched: {len(all_edges)}")

# filters data to N* adsorption
df_all = pd.DataFrame([e['node'] for e in all_edges]).rename(columns={
    'surfaceComposition': 'surface_composition',
    'reactionEnergy':     'delta_GN'
})

mask = (
    (df_all['reactants'] == '{"star": 1, "N2gas": 0.5}') &
    (df_all['products']  == '{"Nstar": 1}')
)
df_cathub = df_all[mask].copy().dropna().reset_index(drop=True)
print(f"N* entries: {len(df_cathub)}")

# drop duplicates and average over several adsorption sites
df_cathub = df_cathub.drop_duplicates(
    subset=['surface_composition', 'facet', 'delta_GN']
).reset_index(drop=True)

df_mean = df_cathub.groupby(
    ['surface_composition', 'facet']
)['delta_GN'].mean().reset_index()

# parse
df_mean['composition'] = df_mean['surface_composition'].apply(safe_composition)
df_mean = df_mean[df_mean['composition'].notna()].reset_index(drop=True)

# featurize (TAKES A WHILE)
magpie_df = pd.DataFrame(
    df_mean['composition'].apply(featurizer.featurize).tolist(),
    columns=featurizer.feature_labels()
)
vo_df = pd.DataFrame(
    df_mean['composition'].apply(lambda c: safe_featurize(vo_featurizer, c)).tolist(),
    columns=vo_featurizer.feature_labels()
)
ao_df = pd.DataFrame(
    df_mean['composition'].apply(lambda c: safe_featurize(ao_featurizer, c)).tolist(),
    columns=ao_featurizer.feature_labels()
)
ao_df = ao_df.apply(pd.to_numeric, errors='coerce')
ao_df = ao_df.loc[:, ao_df.notna().all()]
print(f"ao_df numeric columns retained: {ao_df.shape[1]}")

# fixes training dataframe
df_train = pd.concat(
    [df_mean[['delta_GN']], magpie_df, vo_df, ao_df], axis=1
).dropna()

print(f"Training set shape: {df_train.shape}")
print(f"Features: {df_train.shape[1] - 1}")

# labels data and lower bound, upper bound, and target ΔGN
plt.hist(df_train['delta_GN'], bins=60, edgecolor='k')
plt.axvline(-0.5, color='red',    linestyle='--', label='Target ΔGN = -0.5 eV')
plt.axvline(-1.5, color='orange', linestyle='--', label='Lower bound = -1.5 eV')
plt.axvline( 0.5, color='orange', linestyle='--', label='Upper bound = 0.5 eV')
plt.xlabel('ΔGN (eV)')
plt.ylabel('Count')
plt.title('N* Adsorption Energy Distribution — Mamun 2019')
plt.legend()
plt.show()
print(df_train['delta_GN'].describe())

### Random forest training and evaluation
Trains a random forest regression model on the featurized data pulled from the Catalysis-Hub dataset and displays the quality of the training, both broadly and within the desired range of ΔGN values, with a linear scatterplot.

In [ ]:
X = df_train.drop(columns=['delta_GN']).values
y = df_train['delta_GN'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)

print(f"MAE:  {mae:.4f} eV  (target: < 0.2 eV)")
print(f"R²:   {r2:.4f}     (target: > 0.8)")

plt.scatter(y_test, y_pred, alpha=0.3, s=10)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', label='Perfect prediction')
plt.xlabel('DFT ΔGN (eV)')
plt.ylabel('Predicted ΔGN (eV)')
plt.title('Random Forest Parity Plot — NRR')
plt.legend()
plt.show()

# Performance within screening window
window_mask = (y_test > -1.5) & (y_test < 0.5)
print(f"\nWithin screening window (-1.5 to 0.5 eV):")
print(f"Entries: {window_mask.sum()} / {len(y_test)}")
print(f"MAE: {mean_absolute_error(y_test[window_mask], y_pred[window_mask]):.4f} eV")
print(f"R²:  {r2_score(y_test[window_mask], y_pred[window_mask]):.4f}")

### Predict on HEA candidates and filter
Uses random forest model trained on Catalysis-Hub bimetallic alloy dataset to predict ΔGN of previously-unseen generated alloys.

In [ ]:
# Get exact feature columns from training data — guaranteed to match model
feature_cols = df_train.drop(columns=['delta_GN']).columns.tolist()

# Align candidate features to exactly match training columns
# Any columns in df_featurized not in feature_cols are dropped
# Any missing columns are filled with NaN and those rows dropped
X_candidates = df_featurized.reindex(columns=feature_cols).dropna()
valid_idx = X_candidates.index

df_featurized.loc[valid_idx, 'delta_GN_pred'] = rf.predict(X_candidates.values)

# Filter to NRR viable window
df_candidates = df_featurized.loc[valid_idx].copy()
df_candidates = df_candidates[
    (df_candidates['delta_GN_pred'] > -1.5) &
    (df_candidates['delta_GN_pred'] < 0.5)
].copy()

df_candidates['delta_GN_score'] = (
    df_candidates['delta_GN_pred'] - (-0.25)
).abs()

df_candidates = df_candidates.sort_values(
    'delta_GN_score'
).reset_index(drop=True)

print(f"Candidates in viable NRR window: {len(df_candidates)}")
print(df_candidates[['formula', 'delta_GN_pred', 'delta_GN_score']].head(25))

### Check whether you're going to fucking kill the environment
Uses AlloySustainability to analyze how each element in each compound affects the environmental safety and supply chain risk (among other factors) of the overall compound.

In [ ]:
# Load reference data once
element_indicators = load_element_indicators()
RTHEAs_Fe_df       = load_RTHEAs_vs_Fe_df()
HTHEAs_Ni_df       = load_HTHEAs_vs_Ni_df()

# Compute indicators for each candidate
results = []
for _, row in df_candidates.iterrows():
    mass_fracs = to_mass_fractions(row['elements'])
    impacts    = compute_impacts(mass_fracs, element_indicators)

    # Extract scalar values from any Series objects
    impacts_scalar = {
        k: float(v.iloc[0]) if hasattr(v, 'iloc') else float(v)
        for k, v in impacts.items()
    }
    results.append({
        'formula':  row['formula'],
        'delta_GN': row['delta_GN_pred'],
        **impacts_scalar
    })

df_sustainability = pd.DataFrame(results)
print(f"Scored candidates: {len(df_sustainability)}")
print(df_sustainability.dtypes)
print(df_sustainability.head())


### Render violin plots, download as .zip, show top 25 candidates


In [ ]:
os.makedirs('violin_plots', exist_ok=True)

df_sustainability['delta_GN_abs'] = df_sustainability['delta_GN'].abs()
top25 = df_sustainability.nsmallest(25, 'delta_GN_abs').reset_index(drop=True)

# Save violin plots without displaying
for i, row in top25.iterrows():
    comp    = df_candidates.loc[
        df_candidates['formula'] == row['formula']
    ].iloc[0]['elements']
    fracs   = to_mass_fractions(comp)
    impacts = compute_impacts(fracs, element_indicators)

    fig = plot_alloy_comparison(impacts, RTHEAs_Fe_df, HTHEAs_Ni_df)
    plt.suptitle(
        f"#{i+1}  {row['formula']}  |  ΔGN = {row['delta_GN']:.3f} eV",
        y=1.02
    )
    fig.savefig(
        f"violin_plots/{i+1:02d}_{row['formula']}.png",
        bbox_inches='tight', dpi=150
    )
    plt.close(fig)

print("Violin plots saved.")

with zipfile.ZipFile('violin_plots.zip', 'w') as zf:
    for fname in sorted(os.listdir('violin_plots')):
        zf.write(os.path.join('violin_plots', fname), fname)

display(FileLink('violin_plots.zip'))

# Summary heatmap
indicator_cols = [c for c in top25.columns
                  if c not in ['formula', 'delta_GN', 'delta_GN_abs']]

heatmap_data = top25[indicator_cols].copy()

# Normalize with zero-variance protection
denom = heatmap_data.max() - heatmap_data.min()
denom = denom.replace(0, 1)  # if max == min, division becomes x/1 = x (constant column)
heatmap_data = (heatmap_data - heatmap_data.min()) / denom
heatmap_data.index = [f"#{i+1} {row['formula']}" for i, row in top25.iterrows()]

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(
    heatmap_data,
    annot=top25[indicator_cols].values,
    fmt='.2f',
    cmap='RdYlGn_r',
    linewidths=0.5,
    ax=ax
)
ax.set_title('Sustainability Indicators — Top 25 NRR Candidates (normalized)', pad=12)
ax.set_xlabel('Indicator')
ax.set_ylabel('Candidate')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

### Export results in .csv and calculate composite ranking
Uses a 50-50 split of ΔGN score and sustainability score to find a composite ranking of the list of generated compounds and exports as a downloadable .csv file.

In [ ]:
# --- Merge prediction and sustainability data ---
df_final = df_sustainability.copy()

# For indicators where LOWER is better (all 9 sustainability indicators
# and delta_GN_score), we normalize so that 0 = best, 1 = worst

indicator_cols = [
    'Mass price (USD/kg)',
    'Supply risk',
    'Normalized vulnerability to supply restriction',
    'Embodied energy (MJ/kg)',
    'Water usage (l/kg)',
    'Rock to metal ratio (kg/kg)',
    'Human health damage',
    'Human rights pressure',
    'Labor rights pressure'
]

# Convert to numeric just in case
for col in indicator_cols + ['delta_GN']:
    df_final[col] = pd.to_numeric(df_final[col], errors='coerce')

df_final = df_final.dropna().reset_index(drop=True)

# Normalize sustainability indicators (lower raw value = better = lower normalized)
df_norm = df_final[indicator_cols].copy()
denom = df_norm.max() - df_norm.min()
denom = denom.replace(0, 1)
df_norm = (df_norm - df_norm.min()) / denom

# Normalize delta_GN_score: distance from volcano peak (-0.25 eV)
# Lower distance = better
df_final['delta_GN_score'] = (df_final['delta_GN'] - (-0.25)).abs()
delta_GN_norm = (df_final['delta_GN_score'] - df_final['delta_GN_score'].min()) / (
    df_final['delta_GN_score'].max() - df_final['delta_GN_score'].min()
)

W_ACTIVITY      = 0.50
W_SUSTAIN       = 0.50

sustainability_score = df_norm.mean(axis=1)  # equal weight across 9 indicators
composite_score = (
    W_ACTIVITY * delta_GN_norm +
    W_SUSTAIN  * sustainability_score
)

df_final['sustainability_score'] = sustainability_score
df_final['composite_score']      = composite_score
df_final = df_final.sort_values('composite_score').reset_index(drop=True)
df_final.insert(0, 'rank', df_final.index + 1)

print("Top 25 candidates by composite score:")
print(df_final[['rank', 'formula', 'delta_GN', 'delta_GN_score',
                 'sustainability_score', 'composite_score']].head(25).to_string(index=False))

# --- Export to CSV ---
csv_path = 'HEA_NRR_candidates_ranked.csv'
df_final.to_csv(csv_path, index=False)
print(f"\nSaved {len(df_final)} candidates to {csv_path}")
display(FileLink(csv_path))